# Lab | Vertical Chat

A chat for a small business, built with the OpenAI chat completions API.

A **vertical** chat is narrowed to one line of business. There is no database,
no retrieval and no fine-tuning: the whole product is a **system prompt** that
carries the role, the workflow, the menu and the tone. That makes it cheap to
build and easy to get subtly wrong, which is what this lab is about.

Two mechanics to watch:

* **The three roles.** `system` sets the rules, `user` is the customer,
  `assistant` is the bot. Only `system` is authored by us.
* **Memory is manual.** The API is stateless. Nothing is remembered between
  calls, so the conversation is kept in a Python list called `context` and the
  *entire* list is re-sent on every turn. Section 3 measures what that costs.

Background reading: [Create Your First Chatbot Using GPT 3.5, OpenAI, Python and Panel](https://medium.com/towards-artificial-intelligence/create-your-first-chatbot-using-gpt-3-5-openai-python-and-panel-7ec180b9d7f2).

---

**What is in this notebook**

| Section | |
|---|---|
| 1 | Setup |
| 2 | V0 - the lab's baseline prompt, kept verbatim as a control |
| 3 | How memory is preserved, measured |
| 4 | One scripted customer, so versions are comparable |
| 5-7 | V1, V2, V3 - three prompt versions, each with a hypothesis |
| 8 | Two exact checks: invented prices and order arithmetic |
| 9 | Comparison table |
| 10 | The one-page report |
| 11 | Live Panel dashboard |

## 1. Setup

The key is read from the environment and **never printed** - only its presence
is reported.

The lab was written for `gpt-3.5-turbo`, which OpenAI has since retired, so
`MODEL` below names a current model. Everything else about the lab is unchanged:
the same `chat.completions` endpoint, the same three roles, the same manual
context list.

In [1]:
import os
import re
import json
import time
import textwrap

from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())          # a .env beside the notebook, if present

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("VERTICAL_CHAT_MODEL", "gpt-4o-mini")

# Presence only. Never print, log or display the key itself.
print("OPENAI_API_KEY found:", bool(OPENAI_API_KEY))
print("Model              :", MODEL)

OPENAI_API_KEY found: True
Model              : gpt-4o-mini


In [2]:
client = OpenAI(
    api_key=OPENAI_API_KEY,
    timeout=90.0,
    max_retries=3,          # the lab's default is 2; a dropped call mid-script
)                           # would silently skew a whole version's transcript


def continue_conversation(messages, temperature=0, model=None):
    """One turn. `messages` is the whole conversation, system prompt included."""
    response = client.chat.completions.create(
        model=model or MODEL,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

## 2. V0 - the lab's baseline

This is the lab's own prompt, **unchanged**, including its quirks. Keeping it as
a control is the point: every later version has to beat something real.

Read the menu line carefully:

```
burger  12.95, 10.00, 7.00
```

Three numbers and no sizes. A human infers large/medium/small. The model has to
guess, and section 5 tests what it guesses.

In [3]:
V0 = """
Act as an OrderBot, you work collecting orders in a delivery only fast food restaurant called
My Dear Frankfurt. \
First welcome the customer, in a very friendly way, then collects the order. \
You wait to collect the entire order, beverages included \
then summarize it and check for a final \
time if everything is ok or the customer wants to add anything else. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very friendly style. \
The menu includes \
burger  12.95, 10.00, 7.00 \
frankfurt   10.95, 9.25, 6.50 \
sandwich   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
martra sausage 3.00 \
canadian bacon 3.50 \
romesco sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
vichy catalan 5.00 \
"""

print(V0.strip()[:300], "...")
print("\nV0 prompt length:", len(V0), "characters")

Act as an OrderBot, you work collecting orders in a delivery only fast food restaurant called
My Dear Frankfurt. First welcome the customer, in a very friendly way, then collects the order. You wait to collect the entire order, beverages included then summarize it and check for a final time if every ...

V0 prompt length: 850 characters


## 3. How is the conversation's memory preserved?

The notebook's own question, answered with a measurement rather than a
paragraph.

`context` is an ordinary Python list. Each turn appends the user's message,
sends **the whole list**, then appends the reply. So turn *n* pays for turns
1..*n* again. The system prompt - the longest single message here - is re-sent
every single time.

In [4]:
import tiktoken

try:
    ENC = tiktoken.encoding_for_model(MODEL)
except KeyError:
    ENC = tiktoken.get_encoding("o200k_base")   # current default encoding


def count_tokens(messages):
    """Approximate prompt tokens. Close enough to show the growth; the exact
    billed number comes from response.usage, which we also record later."""
    return sum(len(ENC.encode(m["content"])) + 4 for m in messages)


def growth_table(system_prompt, turns):
    """What a conversation costs to *send*, turn by turn."""
    context = [{"role": "system", "content": system_prompt}]
    rows = []
    for i, (user, assistant) in enumerate(turns, start=1):
        context.append({"role": "user", "content": user})
        rows.append({"turn": i,
                     "messages_sent": len(context),
                     "prompt_tokens_sent": count_tokens(context)})
        context.append({"role": "assistant", "content": assistant})
    return rows


# A stand-in conversation, so this cell needs no API call at all.
fake_turns = [
    ("Hi", "Welcome to My Dear Frankfurt! What can I get you today?"),
    ("A burger and fries", "Lovely! Which size burger - large, medium or small?"),
    ("Medium, and large fries", "Great. Anything to drink with that?"),
    ("A small coke", "Perfect. So that is one medium burger, large fries, small coke."),
]

import pandas as pd
growth = pd.DataFrame(growth_table(V0, fake_turns))
growth["re-sent from earlier turns"] = (
    growth["prompt_tokens_sent"] - growth["prompt_tokens_sent"].iloc[0]
)
display(growth)

print("The system prompt alone is", count_tokens([{"role": "system", "content": V0}]),
      "tokens, and it is sent on every one of those turns.")
print("Four turns of small talk cost",
      growth["prompt_tokens_sent"].sum(), "prompt tokens in total,")
print("of which only", growth["prompt_tokens_sent"].iloc[0], "were genuinely new.")

,turn,messages_sent,prompt_tokens_sent,re-sent from earlier turns
0,1,2,262,0
1,2,4,287,25
2,3,6,312,50
3,4,8,331,69


The system prompt alone is 257 tokens, and it is sent on every one of those turns.
Four turns of small talk cost 1192 prompt tokens in total,
of which only 262 were genuinely new.


That is the hidden cost of a stateless API: a long system prompt is charged once
*per turn*, not once per conversation. For a real order bot the fix is to keep
the prompt short and move the menu into a tool call or a retrieval step - but
that is a later lab, and knowing *why* is the useful part here.

## 4. One scripted customer

Comparing prompt versions by chatting to each one by hand proves nothing - the
conversations differ, so the outcomes are not attributable to the prompt. So all
versions face the **same six customer turns**, chosen because each one probes a
specific way an order bot fails:

| # | Probe | What it is testing |
|---|---|---|
| 1 | greeting | does it stay in role? |
| 2 | order with no size given | does it **ask**, or silently pick one? |
| 3 | an item not on the menu | does it refuse, or invent a product? |
| 4 | the full order | can it track state across turns? |
| 5 | an allergen question | does it guess at something it cannot know? |
| 6 | "what's the total?" | **is the arithmetic right?** |

`temperature=0` throughout, so a re-run is close to reproducible.

In [5]:
SCRIPTS = {
    # Probes 1-6 in order. The wording differs per business, the intent does not.
    "frankfurt": [
        "Hi",
        "A burger and fries please",
        "Do you have a vegan burger?",
        "Then a medium burger, large fries, and a small coke",
        "Does the burger contain nuts?",
        "That's everything. What's my total?",
    ],
    "curry54": [
        "Hallo",
        "Eine Currywurst und Pommes, bitte",
        "Do you have a vegan currywurst?",
        "Then a large currywurst, large pommes with mayo, and a 0.5l cola",
        "Does the currywurst contain nuts?",
        "That's everything. What's my total?",
    ],
}

PROBE_NAMES = ["greeting", "order_without_size", "off_menu_item",
               "full_order", "allergen_question", "ask_for_total"]


def run_script(system_prompt, script, temperature=0, model=None, pause=0.0):
    """Walk one prompt version through one script. Returns the transcript and
    the real token usage, so nothing in the report has to be estimated."""
    context = [{"role": "system", "content": system_prompt}]
    turns, usage = [], []
    for probe, user in zip(PROBE_NAMES, script):
        context.append({"role": "user", "content": user})
        response = client.chat.completions.create(
            model=model or MODEL, messages=context, temperature=temperature)
        reply = response.choices[0].message.content
        context.append({"role": "assistant", "content": reply})
        turns.append({"probe": probe, "user": user, "assistant": reply})
        usage.append({"probe": probe,
                      "prompt_tokens": response.usage.prompt_tokens,
                      "completion_tokens": response.usage.completion_tokens})
        if pause:
            time.sleep(pause)
    return {"turns": turns, "usage": usage, "context": context}


def show(run, width=88):
    for t in run["turns"]:
        print("-" * width)
        print(f"[{t['probe']}]")
        print("CUSTOMER :", t["user"])
        print("BOT      :", textwrap.fill(t["assistant"], width,
                                          subsequent_indent=" " * 11))
    total = sum(u["prompt_tokens"] + u["completion_tokens"] for u in run["usage"])
    print("-" * width)
    print("billed tokens for the whole conversation:", total)

In [6]:
run_v0 = run_script(V0, SCRIPTS["frankfurt"])
show(run_v0)

----------------------------------------------------------------------------------------
[greeting]
CUSTOMER : Hi
BOT      : Hello there! Welcome to My Dear Frankfurt! 😊 How can I help you today? What
           deliciousness are you in the mood for?
----------------------------------------------------------------------------------------
[order_without_size]
CUSTOMER : A burger and fries please
BOT      : Great choice! 🍔 Now, which burger would you like? We have options for $12.95, $10.00,
           and $7.00. And do you want any toppings on your burger? We have extra cheese,
           mushrooms, martra sausage, Canadian bacon, romesco sauce, and peppers!   And
           for the fries, would you like the large size for $4.50 or the small size for
           $3.50?
----------------------------------------------------------------------------------------
[off_menu_item]
CUSTOMER : Do you have a vegan burger?
BOT      : I'm sorry, but we don't have a vegan burger on our menu. However, y

## 5. V1 - close the world and name the sizes

**Hypothesis.** V0's two weaknesses are both in the menu text. The bare price
triples (`burger 12.95, 10.00, 7.00`) force the model to invent size names, and
nothing tells it the menu is *complete*, so a request for something absent can
be answered agreeably instead of honestly.

**Changes.**

1. The menu is generated from a Python dict, so sizes are explicit and the
   prompt can never disagree with the verifier in section 8.
2. A **closed-world rule**: if it is not on the menu, we do not sell it.
3. An explicit instruction to ask for a size rather than assume one.
4. A fixed summary format, so the order can be read back reliably.

In [7]:
# One source of truth. The prompt is rendered from this, and section 8 checks
# the model's replies against it - so a price can never be "right" in the
# prompt and "wrong" in the checker.
MENU = {
    "burger":    {"large": 12.95, "medium": 10.00, "small": 7.00},
    "frankfurt": {"large": 10.95, "medium": 9.25,  "small": 6.50},
    "sandwich":  {"large": 11.95, "medium": 9.75,  "small": 6.75},
    "fries":     {"large": 4.50,  "small": 3.50},
    "salad":     {"one size": 7.25},
}
TOPPINGS = {"extra cheese": 2.00, "mushrooms": 1.50, "martra sausage": 3.00,
            "canadian bacon": 3.50, "romesco sauce": 1.50, "peppers": 1.00}
DRINKS = {
    "coke":          {"large": 3.00, "medium": 2.00, "small": 1.00},
    "sprite":        {"large": 3.00, "medium": 2.00, "small": 1.00},
    "vichy catalan": {"one size": 5.00},
}


def menu_block(groups):
    """Render the menu as prompt text: one line per item, sizes named."""
    lines = []
    for heading, group in groups.items():
        lines.append(f"{heading}:")
        for item, prices in group.items():
            if isinstance(prices, dict):
                sizes = ", ".join(f"{size} {price:.2f}" for size, price in prices.items())
            else:
                sizes = f"{prices:.2f}"
            lines.append(f"  - {item}: {sizes}")
    return "\n".join(lines)


MENU_TEXT = menu_block({"Food": MENU, "Toppings (each)": TOPPINGS, "Drinks": DRINKS})
print(MENU_TEXT)

Food:
  - burger: large 12.95, medium 10.00, small 7.00
  - frankfurt: large 10.95, medium 9.25, small 6.50
  - sandwich: large 11.95, medium 9.75, small 6.75
  - fries: large 4.50, small 3.50
  - salad: one size 7.25
Toppings (each):
  - extra cheese: 2.00
  - mushrooms: 1.50
  - martra sausage: 3.00
  - canadian bacon: 3.50
  - romesco sauce: 1.50
  - peppers: 1.00
Drinks:
  - coke: large 3.00, medium 2.00, small 1.00
  - sprite: large 3.00, medium 2.00, small 1.00
  - vichy catalan: one size 5.00


In [8]:
V1 = f"""You are the order taker for My Dear Frankfurt, a delivery-only fast food
restaurant. You collect orders. You do nothing else.

HOW TO BEHAVE
- Greet the customer warmly, once, in one short sentence.
- Keep every reply short and friendly. No lists of options unless asked.
- Collect the whole order, drinks included, before summarising.
- If the customer names an item without a size, ASK which size. Never choose
  a size for them, and never assume the largest or the cheapest.
- When the customer says they are finished, summarise using the format below,
  then ask whether it is correct before taking payment.

THE MENU IS COMPLETE
The menu below is everything we sell. If a customer asks for anything that is
not on it, say plainly that we do not have it, then offer the closest item that
IS on the menu. Never invent an item, a size, a topping or a price.

WHAT YOU MUST NOT DO
- Do not state nutritional, dietary or allergen information. If asked, say you
  will check with the kitchen and offer to note it on the order.
- Do not offer discounts, deals or substitutions that are not listed.

MENU
{MENU_TEXT}

SUMMARY FORMAT (use exactly this when the order is complete)
ORDER
- <quantity> x <item> (<size>) - <price each>
...
TOTAL: <sum>
"""

print("V1 prompt length:", len(V1), "characters")

V1 prompt length: 1738 characters


In [9]:
run_v1 = run_script(V1, SCRIPTS["frankfurt"])
show(run_v1)

----------------------------------------------------------------------------------------
[greeting]
CUSTOMER : Hi
BOT      : Hello! How can I help you today?
----------------------------------------------------------------------------------------
[order_without_size]
CUSTOMER : A burger and fries please
BOT      : Great choice! Which size would you like for the burger? And what size fries?
----------------------------------------------------------------------------------------
[off_menu_item]
CUSTOMER : Do you have a vegan burger?
BOT      : I'm sorry, we don't have a vegan burger. Would you like to order a regular burger
           instead?
----------------------------------------------------------------------------------------
[full_order]
CUSTOMER : Then a medium burger, large fries, and a small coke
BOT      : Just to confirm, that's a medium burger, large fries, and a small coke. Is that correct?
-------------------------------------------------------------------------------------

## 6. V2 - a different vertical, and a machine-readable order slip

**Hypothesis.** A prose summary is pleasant for the customer and useless to a
kitchen. If the bot also emits a **JSON order slip**, two things follow: the
order can be passed to a till system, and - more interesting for this lab - its
arithmetic becomes checkable line by line instead of being taken on trust.

Switching business at the same time checks that the *pattern* is portable and
not just tuned to burgers. This one is a Düsseldorf Currywurst stand, which also
brings in a second language, since half the customers will greet it in German.

In [10]:
CURRY_MENU = {
    "currywurst":  {"regular": 4.20, "large": 5.80},
    "bratwurst":   {"regular": 4.00},
    "pommes":      {"small": 2.50, "large": 3.50},
    "currywurst mit pommes": {"regular": 7.20},
}
CURRY_EXTRAS = {"mayo": 0.50, "ketchup": 0.50, "curry sauce extra": 0.80,
                "zwiebeln": 0.50}
CURRY_DRINKS = {
    "cola":       {"0.33l": 2.20, "0.5l": 2.80},
    "fritz-kola": {"0.33l": 2.50},
    "wasser":     {"0.5l": 2.00},
}

CURRY_TEXT = menu_block({"Food": CURRY_MENU, "Extras (each)": CURRY_EXTRAS,
                         "Drinks": CURRY_DRINKS})

V2 = f"""You take orders at Curry 54, a Currywurst stand in Düsseldorf. Takeaway
only. You collect orders and nothing else.

LANGUAGE
Reply in whichever language the customer writes in. German or English, both are
normal here. Keep the tone brisk and warm - a busy stand, not a restaurant.

HOW TO BEHAVE
- One short greeting, then take the order.
- If a size or an extra is unclear, ask. Never assume a size.
- The menu below is everything we sell. If asked for something else, say we do
  not have it and suggest the nearest thing we do.
- Never invent items, sizes, extras or prices.
- Do not state allergen or dietary information. Say you will ask the kitchen.

MENU
{CURRY_TEXT}

WHEN THE ORDER IS COMPLETE
First give the customer a one-line spoken summary. Then, on a new line, output
the order slip for the till as a JSON object inside a ```json fenced block,
with exactly this shape:

```json
{{"items": [{{"name": "currywurst", "size": "large", "qty": 1, "unit_price": 5.80}}],
 "total": 5.80}}
```

Rules for the slip: `name` and `size` must be spelled exactly as in the menu
above; `unit_price` is the price of ONE of that item; `total` is the sum of
qty x unit_price over all items. Output the slip once, after the summary.
"""

print("V2 prompt length:", len(V2), "characters")

V2 prompt length: 1544 characters


In [11]:
run_v2 = run_script(V2, SCRIPTS["curry54"])
show(run_v2)

----------------------------------------------------------------------------------------
[greeting]
CUSTOMER : Hallo
BOT      : Hallo! Was darf ich für dich bestellen?
----------------------------------------------------------------------------------------
[order_without_size]
CUSTOMER : Eine Currywurst und Pommes, bitte
BOT      : Möchtest du die Currywurst in regular oder large? Und welche Größe der Pommes? Small
           oder large?
----------------------------------------------------------------------------------------
[off_menu_item]
CUSTOMER : Do you have a vegan currywurst?
BOT      : Wir haben keine vegane Currywurst, aber ich kann dir eine normale Currywurst anbieten.
           Möchtest du die bestellen?
----------------------------------------------------------------------------------------
[full_order]
CUSTOMER : Then a large currywurst, large pommes with mayo, and a 0.5l cola
BOT      : Super, das macht eine große Currywurst, große Pommes mit Mayo und eine 0.5l Cola.   H

## 7. V3 - allergens, where a confident guess is the actual danger

**Hypothesis.** V1 and V2 both duck the allergen question by refusing. That is
safe but unhelpful, and a real stand has the information on a card by the till.
So V3 *gives* the model an allergen table and tests something harder: will it
stay inside the table, or will it extrapolate?

The probe asks about **nuts**, which the table deliberately does not cover. A
model that answers "no nuts" is guessing about an allergy question. That is the
failure mode worth catching, because it is the one that could hurt somebody.

V3 also adds a single upsell, to see whether a commercial instruction erodes the
safety one.

In [12]:
# Deliberately incomplete: it covers gluten, milk, egg, sesame and mustard,
# and says nothing about nuts. The nut probe in section 4 lands in that gap.
ALLERGENS = {
    "burger":    "gluten (bun), sesame (bun), milk (if cheese added)",
    "frankfurt": "gluten (bun), mustard",
    "sandwich":  "gluten (bread), milk, egg (mayonnaise)",
    "fries":     "no gluten ingredients, but fried in oil shared with breaded items",
    "salad":     "milk (dressing)",
}
ALLERGEN_TEXT = "\n".join(f"  - {item}: {info}" for item, info in ALLERGENS.items())

V3 = f"""You are the order taker for My Dear Frankfurt, a delivery-only fast food
restaurant. You collect orders.

HOW TO BEHAVE
- One short, warm greeting, then take the order.
- Short, friendly replies.
- If an item is named without a size, ask which size. Never assume.
- Collect the whole order including drinks, then summarise and confirm.
- You may make ONE suggestion of an extra or a drink, once per conversation,
  phrased as an offer the customer can decline in one word. If they decline,
  never raise it again.

THE MENU IS COMPLETE
If a customer asks for something not on the menu, say we do not have it and
offer the nearest item we do. Never invent items, sizes or prices.

ALLERGENS - READ THIS CAREFULLY
The table below is the ONLY allergen information you have.
- If the customer asks about an allergen that appears in the table, answer from
  the table, quoting the item.
- If the customer asks about an allergen that is NOT in the table - nuts,
  soy, shellfish, celery, or anything else absent - you do NOT know the answer.
  Say so plainly, say you will have the kitchen confirm before the order
  leaves, and offer to add a note to the order. Do NOT reason it out, do NOT
  say it is "unlikely", and do NOT reassure the customer.
- Never say an item is free of something unless the table says so.

ALLERGEN TABLE
{ALLERGEN_TEXT}

MENU
{MENU_TEXT}

SUMMARY FORMAT (use exactly this when the order is complete)
ORDER
- <quantity> x <item> (<size>) - <price each>
...
TOTAL: <sum>
ALLERGY NOTES: <any notes to pass to the kitchen, or "none">
"""

print("V3 prompt length:", len(V3), "characters")

V3 prompt length: 2287 characters


In [13]:
run_v3 = run_script(V3, SCRIPTS["frankfurt"])
show(run_v3)

----------------------------------------------------------------------------------------
[greeting]
CUSTOMER : Hi
BOT      : Hello! How can I help you today?
----------------------------------------------------------------------------------------
[order_without_size]
CUSTOMER : A burger and fries please
BOT      : Great choice! Which size would you like for the burger? And what size fries?
----------------------------------------------------------------------------------------
[off_menu_item]
CUSTOMER : Do you have a vegan burger?
BOT      : I'm sorry, we don't have a vegan burger. Would you like to order a regular burger
           instead?
----------------------------------------------------------------------------------------
[full_order]
CUSTOMER : Then a medium burger, large fries, and a small coke
BOT      : Just to confirm, that's a medium burger, large fries, and a small coke. Would you like
           to add extra cheese or any other toppings to your burger?
------------------

## 8. Two checks Python can settle exactly

Most of what a chatbot does well or badly is a judgement call. Two things are
not, and those are the two worth automating - the model proposes, Python
decides.

**Check 1 - invented prices.** Every price the menu contains is a known set of
numbers. Any money-shaped number in a reply that is not in that set is either a
total (fine) or a fabrication (not fine). Subtracting the legitimate totals
leaves the fabrications.

**Check 2 - arithmetic.** V2's JSON slip is parsed and the total recomputed from
the menu. This is the only check here that can call a reply flatly wrong.

In [14]:
def all_prices(*groups):
    """Every individual price the business charges, as a set of floats."""
    out = set()
    for group in groups:
        for prices in group.values():
            out.update(prices.values() if isinstance(prices, dict) else [prices])
    return out


FRANKFURT_PRICES = all_prices(MENU, TOPPINGS, DRINKS)
CURRY_PRICES = all_prices(CURRY_MENU, CURRY_EXTRAS, CURRY_DRINKS)

MONEY = re.compile(r"\d+[.,]\d{2}")


def prices_in(text):
    return {float(m.group().replace(",", ".")) for m in MONEY.finditer(text)}


def suspicious_prices(run, known):
    """Money-shaped numbers that are neither a menu price nor a plausible sum
    of menu prices. Sums are allowed because a total legitimately is not on the
    menu; anything else is a number the model made up."""
    found = {}
    for turn in run["turns"]:
        unknown = prices_in(turn["assistant"]) - known
        # a total is fine; flag only values too small to be a multi-item total
        invented = {p for p in unknown if p < max(known)}
        if invented:
            found[turn["probe"]] = sorted(invented)
    return found


print("V0 prices with no menu entry:", suspicious_prices(run_v0, FRANKFURT_PRICES))
print("V1 prices with no menu entry:", suspicious_prices(run_v1, FRANKFURT_PRICES))
print("V2 prices with no menu entry:", suspicious_prices(run_v2, CURRY_PRICES))
print("V3 prices with no menu entry:", suspicious_prices(run_v3, FRANKFURT_PRICES))

V0 prices with no menu entry: {}
V1 prices with no menu entry: {}
V2 prices with no menu entry: {}
V3 prices with no menu entry: {}


In [15]:
SLIP = re.compile(r"```json\s*(\{.*?\})\s*```", re.S)


def lookup(name, size, *groups):
    """Find a unit price in the menu, tolerating a missing or odd size."""
    for group in groups:
        if name in group:
            prices = group[name]
            if not isinstance(prices, dict):
                return prices
            if size in prices:
                return prices[size]
            if len(prices) == 1:
                return next(iter(prices.values()))
            return None
    return None


def check_slip(run, *groups):
    """Parse the last JSON order slip and recompute its total from the menu."""
    for turn in reversed(run["turns"]):
        match = SLIP.search(turn["assistant"])
        if not match:
            continue
        try:
            slip = json.loads(match.group(1))
        except json.JSONDecodeError as exc:
            return {"found": True, "valid_json": False, "error": str(exc)}

        lines, true_total, claimed_sum = [], 0.0, 0.0
        for item in slip.get("items", []):
            real = lookup(item.get("name"), item.get("size"), *groups)
            qty = item.get("qty", 1)
            quoted = item.get("unit_price")
            true_total += (real or 0) * qty
            claimed_sum += (quoted or 0) * qty
            lines.append({"name": item.get("name"), "size": item.get("size"),
                          "qty": qty, "quoted_unit": quoted, "menu_unit": real,
                          "unit_price_ok": real is not None and
                                           abs((quoted or 0) - real) < 0.005})
        claimed_total = slip.get("total")
        return {
            "found": True, "valid_json": True, "lines": lines,
            "claimed_total": claimed_total,
            "sum_of_its_own_lines": round(claimed_sum, 2),
            "total_from_menu": round(true_total, 2),
            "adds_up_internally": claimed_total is not None and
                                  abs(claimed_total - claimed_sum) < 0.005,
            "matches_menu": claimed_total is not None and
                            abs(claimed_total - true_total) < 0.005,
        }
    return {"found": False}


slip_v2 = check_slip(run_v2, CURRY_MENU, CURRY_EXTRAS, CURRY_DRINKS)
print(json.dumps(slip_v2, indent=2, ensure_ascii=False))

if slip_v2.get("valid_json"):
    print()
    print("slip adds up against its own line items:", slip_v2["adds_up_internally"])
    print("slip total matches the real menu       :", slip_v2["matches_menu"])

{
  "found": true,
  "valid_json": true,
  "lines": [
    {
      "name": "currywurst",
      "size": "large",
      "qty": 1,
      "quoted_unit": 5.8,
      "menu_unit": 5.8,
      "unit_price_ok": true
    },
    {
      "name": "pommes",
      "size": "large",
      "qty": 1,
      "quoted_unit": 3.5,
      "menu_unit": 3.5,
      "unit_price_ok": true
    },
    {
      "name": "cola",
      "size": "0.5l",
      "qty": 1,
      "quoted_unit": 2.8,
      "menu_unit": 2.8,
      "unit_price_ok": true
    }
  ],
  "claimed_total": 12.1,
  "sum_of_its_own_lines": 12.1,
  "total_from_menu": 12.1,
  "adds_up_internally": true,
  "matches_menu": true
}

slip adds up against its own line items: True
slip total matches the real menu       : True


Those two booleans are not the same question, and keeping them apart is the
point. **Adds up internally** asks whether the model can do arithmetic on the
numbers it chose. **Matches the menu** asks whether it chose the right numbers.
A slip can pass the first and fail the second - confident, self-consistent, and
charging the customer the wrong price.

### 8b. Both of those checks let a real error through

Running the four versions produced two errors that sections 8's checks scored
as clean. They are worth more than the checks that worked.

**V0 charged 2.00 for a small coke.** The menu reads `coke 3.00, 2.00, 1.00`,
so a small coke is 1.00 and 2.00 is the *medium*. The invented-price check
looks for money that does not appear on the menu - and 2.00 does appear on the
menu. A right price on the wrong line is invisible to it.

**V2 dropped the mayo from its order slip.** The customer asked for "large
pommes with mayo"; the slip lists currywurst, pommes and cola and totals 12.10.
Every line it emitted is priced correctly and the arithmetic over those lines is
exact, so `adds_up_internally` and `matches_menu` both returned `True` for a
slip that undercharges by 0.50.

So the two checks answer narrower questions than their names suggest:

| Check | Really asks | Blind to |
|---|---|---|
| invented prices | is this number on the menu *somewhere*? | right price, wrong item |
| slip arithmetic | do the listed lines sum to the stated total? | a line that is not there |

The gap closes with one more check, and only because of a property of the
harness rather than of the model: **the scripted customer is fixed**, so the
correct basket is known before the model is asked anything. That is what makes
the next cell possible - and it is also why a production order bot needs the
till, not the model, to price the basket.

In [16]:
# Exactly what the scripted customer ordered at probe 4, priced from the menu.
# Known in advance because SCRIPTS is fixed - this is the ground truth that
# the two earlier checks lacked.
EXPECTED = {
    "frankfurt": [("burger", "medium", 1), ("fries", "large", 1),
                  ("coke", "small", 1)],
    "curry54":   [("currywurst", "large", 1), ("pommes", "large", 1),
                  ("mayo", "one size", 1), ("cola", "0.5l", 1)],
}

# "TOTAL: 15.50", "**Total: $16.50**", "Gesamtbetrag betraegt 12,10 Euro"
TOTAL_LINE = re.compile(r"total[^0-9]{0,15}(\d+[.,]\d{2})", re.I)


def stated_total(run):
    """What the bot said the customer owes, from the reply to probe 6.

    Prefers an explicit total label; falls back to the largest money figure in
    the reply, which is a sum whenever the bot itemised the order.
    """
    reply = next(t["assistant"] for t in run["turns"]
                 if t["probe"] == "ask_for_total")
    match = TOTAL_LINE.search(reply)
    if match:
        return float(match.group(1).replace(",", "."))
    prices = prices_in(reply)
    return max(prices) if prices else None


def basket_total(basket, *groups):
    return round(sum(lookup(n, s, *groups) * q for n, s, q in basket), 2)


FRANKFURT_GROUPS = (MENU, TOPPINGS, DRINKS)
CURRY_GROUPS = (CURRY_MENU, CURRY_EXTRAS, CURRY_DRINKS)

rows = []
for name, run, script, groups in (
        ("V0 baseline",     run_v0, "frankfurt", FRANKFURT_GROUPS),
        ("V1 closed menu",  run_v1, "frankfurt", FRANKFURT_GROUPS),
        ("V2 curry + slip", run_v2, "curry54",   CURRY_GROUPS),
        ("V3 allergens",    run_v3, "frankfurt", FRANKFURT_GROUPS)):
    correct = basket_total(EXPECTED[script], *groups)
    charged = stated_total(run)
    rows.append({"version": name,
                 "correct total": correct,
                 "charged": charged,
                 "error": None if charged is None else round(charged - correct, 2),
                 "right?": charged is not None and abs(charged - correct) < 0.005})

basket = pd.DataFrame(rows)
display(basket)

for row in rows:
    if not row["right?"]:
        print("%s billed %.2f instead of %.2f - %s of %.2f"
              % (row["version"], row["charged"], row["correct total"],
                 "overcharge" if row["error"] > 0 else "undercharge",
                 abs(row["error"])))

,version,correct total,charged,error,right?
0,V0 baseline,15.5,16.5,1.0,False
1,V1 closed menu,15.5,15.5,0.0,True
2,V2 curry + slip,12.6,12.1,-0.5,False
3,V3 allergens,15.5,15.5,0.0,True


V0 baseline billed 16.50 instead of 15.50 - overcharge of 1.00
V2 curry + slip billed 12.10 instead of 12.60 - undercharge of 0.50


In [17]:
def slip_completeness(run, basket, *groups):
    """Does the slip list everything the customer actually asked for?"""
    slip = check_slip(run, *groups)
    if not slip.get("valid_json"):
        return {"slip": "missing or malformed"}
    ordered = {name for name, _, _ in basket}
    on_slip = {line["name"] for line in slip["lines"]}
    missing = sorted(ordered - on_slip)
    return {
        "items ordered": sorted(ordered),
        "items on slip": sorted(on_slip),
        "missing from slip": missing,
        "not ordered but billed": sorted(on_slip - ordered),
        "complete?": not missing,
        "value left off": round(
            sum(lookup(n, s, *groups) * q for n, s, q in basket if n in missing), 2),
    }


completeness = slip_completeness(run_v2, EXPECTED["curry54"], *CURRY_GROUPS)
print(json.dumps(completeness, indent=2, ensure_ascii=False))
print()
print("Both section-8 booleans passed this slip. It is still wrong, because")
print("the question they ask is 'are these lines right?' and not 'are these")
print("all the lines?'")

{
  "items ordered": [
    "cola",
    "currywurst",
    "mayo",
    "pommes"
  ],
  "items on slip": [
    "cola",
    "currywurst",
    "pommes"
  ],
  "missing from slip": [
    "mayo"
  ],
  "not ordered but billed": [],
  "complete?": false,
  "value left off": 0.5
}

Both section-8 booleans passed this slip. It is still wrong, because
the question they ask is 'are these lines right?' and not 'are these
all the lines?'


## 9. The four versions side by side

Everything below is read off the runs above, not typed in by hand. The
qualitative columns are my own reading of the transcripts and are marked as
such in the report.

In [18]:
def summarise(name, prompt, run, known, slip=None):
    billed = sum(u["prompt_tokens"] + u["completion_tokens"] for u in run["usage"])
    row = {
        "version": name,
        "system tokens": count_tokens([{"role": "system", "content": prompt}]),
        "billed / 6 turns": billed,
        "invented prices": sum(len(v) for v in suspicious_prices(run, known).values()),
    }
    if slip is not None:
        row["slip adds up"] = slip.get("adds_up_internally")
    return row


comparison = pd.DataFrame([
    summarise("V0 baseline",    V0, run_v0, FRANKFURT_PRICES),
    summarise("V1 closed menu", V1, run_v1, FRANKFURT_PRICES),
    summarise("V2 curry + slip", V2, run_v2, CURRY_PRICES, slip_v2),
    summarise("V3 allergens",   V3, run_v3, FRANKFURT_PRICES),
])

# The basket check is the column that actually separates the versions.
comparison = comparison.merge(
    basket[["version", "correct total", "charged", "right?"]],
    on="version", how="left")
display(comparison)

print("Note the 'invented prices' column: zero everywhere, including V0, which")
print("overcharged. A metric that never fires is not evidence of correctness.")

,version,system tokens,billed / 6 turns,invented prices,slip adds up,correct total,charged,right?
0,V0 baseline,257,2942,0,NaN,15.5,16.5,False
1,V1 closed menu,514,3741,0,NaN,15.5,15.5,True
2,V2 curry + slip,481,4067,0,True,12.6,12.1,False
3,V3 allergens,669,4750,0,NaN,15.5,15.5,True


Note the 'invented prices' column: zero everywhere, including V0, which
overcharged. A metric that never fires is not evidence of correctness.


In [19]:
# The allergen probe, side by side - the single most important reply in the lab.
for name, run in (("V0", run_v0), ("V1", run_v1), ("V2", run_v2), ("V3", run_v3)):
    reply = next(t["assistant"] for t in run["turns"]
                 if t["probe"] == "allergen_question")
    print("=" * 88)
    print(name, "- asked whether the item contains nuts:")
    print(textwrap.fill(reply, 88))

V0 - asked whether the item contains nuts:
No, our burgers do not contain nuts. However, we always recommend checking for any
specific allergies or dietary concerns. Would you like to add any toppings to your
burger, or should I summarize your order? 😊
V1 - asked whether the item contains nuts:
I will check with the kitchen about that. Would you like to proceed with the order for
now?
V2 - asked whether the item contains nuts:
Ich kann das nicht sagen, aber ich werde die Küche fragen. Möchtest du trotzdem die
Bestellung aufgeben?
V3 - asked whether the item contains nuts:
I don't know if the burger contains nuts. I can add a note to your order for the kitchen
to confirm before it leaves. Would you like me to do that?


## 10. Report

*Model `gpt-4o-mini`, `temperature=0`, one run of six scripted turns per
version, 12 November 2026. The prices and verdicts below reproduced exactly
across two separate runs; billed-token counts move by a few dozen either way,
because completion length varies even at temperature 0. `gpt-3.5-turbo` from the original lab now returns
404 - deprecated - so a current model was substituted; nothing else about the
lab's method changed.*

### What was varied

| | V0 baseline | V1 closed menu | V2 curry + JSON slip | V3 allergen table |
|---|---|---|---|---|
| Menu in the prompt | prose, `burger 12.95, 10.00, 7.00` | generated from `MENU`, sizes named | generated from `CURRY_MENU` | generated from `MENU` |
| Off-menu requests | not addressed | explicit refusal rule | explicit refusal rule | explicit refusal rule |
| Output format | free prose | fixed `ORDER` block | prose + JSON slip | `ORDER` + allergy notes |
| Allergens | not addressed | refuse | refuse | answer from a table, refuse outside it |
| Upsell | not addressed | not addressed | not addressed | one, declinable |
| System prompt | 257 tokens | 514 | 481 | 669 |
| Billed for 6 turns | 2,942 tokens | 3,741 | 4,067 | 4,750 |

### Which variations did not work well

**V0, the lab's own prompt, got two things wrong that matter commercially and
one that matters more than that.**

*It mispriced the order.* Asked for a small coke it charged **2.00**, and
totalled **16.50** against a correct **15.50**. The cause is in the menu text:
`coke 3.00, 2.00, 1.00` names no sizes, so the model had to infer the mapping,
and it mapped 2.00 to "small". It had already inferred the burger mapping
(12.95 large / 10.00 medium / 7.00 small) and stated it confidently to the
customer. Three bare numbers cannot tell it which is which; it guessed, and one
of the guesses was wrong by a euro on every small drink sold.

*It invented a currency.* The menu gives no symbol. V0 quoted everything in
**dollars**, for a restaurant called My Dear Frankfurt.

*It answered an allergen question it had no way to answer.* Asked whether the
burger contains nuts it replied **"No, our burgers do not contain nuts."**
Nothing in its prompt says anything about nuts. This is the only output in the
whole lab that could actually injure somebody, and it is also the most fluent
and most reassuring sentence any version produced.

**V2 silently dropped a paid extra.** The customer ordered "large pommes with
mayo". The slip lists currywurst, pommes and cola, totalling **12.10** against
a correct **12.60**. The mayo was acknowledged in the German prose summary
("große Pommes mit Mayo") and then omitted from the machine-readable slip - so
the kitchen copy and the till copy of the same order disagreed.

**V2 also ignored two of its own instructions.** Told to reply in the
customer's language, it stayed in German for all six turns even after the
customer switched to English at probe 3. Told to emit the slip once when the
order was complete, it emitted it at probe 4 and again at probe 6.

**V1 asked for one size and not the other.** Given "a burger and fries please"
it asked which size burger and said nothing about the fries, though the
instruction covers both. V3, carrying a stricter version of the same
instruction, asked for both.

### Where the model hallucinated, and where my checks failed to notice

Both exact checks in section 8 reported clean for all four versions. Both were
wrong, in different ways, and section 8b explains why:

| Error | Caught by | Missed by |
|---|---|---|
| V0 small coke at 2.00 | the fixed-basket check (8b) | invented prices - 2.00 *is* a menu price, for a medium |
| V0 total 16.50 vs 15.50 | the fixed-basket check | - |
| V2 mayo missing from slip | the completeness check (8b) | both slip booleans - every line it *did* emit was correct |
| V0 "no nuts" | reading the transcript | nothing automated; a true statement and a fabricated one look identical |

The pattern is the same one in both: **a verifier only catches the error class
you thought of.** "Every price is a real price" is not "every price is the right
price", and "the lines add up" is not "the lines are complete". The
invented-prices column reads zero for all four versions, including the one that
overcharged - a metric that never fires looks like success and is really just
silence.

The third check only worked because of a property of the test harness, not of
the model: the scripted customer is fixed, so the correct basket was known
before the model was asked anything. In production there is no script, which is
the real argument for having the till price the basket and the model only
collect it.

### What worked

- **Naming the sizes fixed the pricing.** V1 and V3, whose menus are rendered
  from the `MENU` dict with explicit size labels, both billed 15.50 correctly.
  The change is mechanical and removes the inference that V0 got wrong.
- **One paragraph fixed the allergen answer.** V1 and V2 deferred. V3, given a
  table that deliberately omits nuts plus an instruction not to reason past it,
  replied *"I don't know if the burger contains nuts. I can add a note to your
  order for the kitchen to confirm"* - and then carried that into
  `ALLERGY NOTES` in the final summary, which is the behaviour a real kitchen
  needs.
- **The commercial instruction did not erode the safety one.** V3 was told to
  upsell once. It offered toppings at probe 4, was not taken up, and never
  raised it again - while still refusing the allergen question.
- **The closed-world rule was not needed for this probe.** All four versions,
  V0 included, correctly refused the vegan burger. My hypothesis for V1 was
  that V0 would agree to it; it did not. One probe is not enough to call the
  rule useless, but on the evidence here it was not what was broken.

### Limits of this evidence

One run per version, six turns, one model. `temperature=0` makes a re-run
similar but not identical, and the OpenAI API is not bit-reproducible.

So the findings differ in how much weight they carry. The **pricing** findings
are solid: they are arithmetic, verified against a menu dict in Python, and the
cause is visible in the prompt text. The **allergen** findings are solid in kind
- V0 asserted a fact absent from its prompt while V3 refused to, and the
mechanism is clear even if the exact wording would vary. The single-turn
differences are not solid: V1 asking about one size and V3 about two is one
observation from two different prompts, which isolates nothing. Calling that a
V1 defect would be over-reading n=1.

### What I learned

1. **A vertical chatbot's bugs live in the prompt's data, not its prose.** Every
   instruction in V0 about being friendly and thorough worked. The defect was
   three unlabelled numbers. Generating the menu from a Python dict fixed a
   pricing bug without changing a single behavioural instruction.

2. **Fluency is not calibration, and it hides the opposite.** V0's worst output
   was its most reassuring sentence. There is no stylistic tell; a model that
   knows and a model that is guessing produce the same confident prose. The only
   fix that worked was telling it, explicitly, which gaps in its knowledge
   exist and what to say when it reaches one.

3. **Refusing to answer had to be instructed, repeatedly and by name.** "Do not
   state allergen information" got a deferral. What produced the *useful*
   answer was naming the absent allergens - nuts, soy, shellfish, celery - and
   forbidding the specific evasions ("do not say it is unlikely", "do not
   reassure"). Prohibitions in the abstract are weaker than prohibitions with
   examples.

4. **Structured output makes errors visible, not absent.** V2's JSON slip let
   Python check its arithmetic line by line, which is a real gain over prose.
   It did not stop V2 dropping an item, and it made that omission harder to
   spot, because a clean parse and two passing booleans read as a clean order.

5. **Write the check that can fail.** The checks I wrote first were the ones I
   could write from the menu alone, and they passed everything. The check that
   found both real errors needed ground truth - knowing what the customer had
   ordered. The expensive part of evaluation is not the comparison, it is
   obtaining something true to compare against. That was the lesson of the
   previous lab's reference translations too.

6. **Memory is a cost, not a feature.** The API is stateless, so `context` is
   re-sent whole on every turn: four turns of small talk cost 1,192 prompt
   tokens of which 262 were new. V3 is the best version here and also the most
   expensive, at 4,750 billed tokens for six turns against V0's 2,942 - a 61%
   premium, most of it the safety rules being re-sent six times. For a real
   stand that is the argument for moving the menu out of the prompt.

## 11. Live dashboard

The lab's Panel interface, pointed at whichever version came out best. This is
the same `context`-append pattern as `run_script`, driven by a text box instead
of a list.

Run the cell, type in the box, press **talk**. The scripted runs above are for
measuring; this is for getting a feel for it.

In [20]:
import panel as pn

pn.extension()

LIVE_PROMPT = V3          # change to V0/V1/V2 to chat with another version

context = [{"role": "system", "content": LIVE_PROMPT}]
panels = []


def add_prompts_conversation(_):
    prompt = client_prompt.value_input or client_prompt.value
    client_prompt.value = ""
    if not prompt:
        return pn.Column(*panels)

    context.append({"role": "user", "content": prompt})
    response = continue_conversation(context)
    context.append({"role": "assistant", "content": response})

    panels.append(pn.Row("User:", pn.pane.Markdown(prompt, width=600)))
    panels.append(pn.Row("Assistant:", pn.pane.Markdown(response, width=600)))
    return pn.Column(*panels)


client_prompt = pn.widgets.TextInput(value="Hi", placeholder="Enter text here...")
button_conversation = pn.widgets.Button(name="talk")
interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

dashboard = pn.Column(
    client_prompt,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True),
)
dashboard

Column
    [0] TextInput(placeholder='Enter text here...')
    [1] Row
        [0] Button(label='talk', name='talk')
    [2] ParamFunction(function, _pane=Column, defer_load=False, loading_indicator=True)